In [1]:
import os
import pdfplumber

In [2]:
DATA_PATH = os.path.join("samples")

def get_pdf_paths(path):
    pdf_paths = []
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dir not found: {DATA_PATH}")

    for file in os.listdir(path):
        if file.endswith(".pdf"):
            pdf_paths.append(os.path.join(path, file))

    print(f"Znaleziono pliki: {pdf_paths}")
    return pdf_paths


In [3]:
pdf_paths = get_pdf_paths(DATA_PATH)
pdf_paths

Znaleziono pliki: ['samples\\karta_info_skargi_wnioski_sn_233c5c16.pdf', 'samples\\Mapa_Otwartych_Zasobow_Edukacyjnych_93594bec.pdf', 'samples\\OC_os_fiz_przy_EDU_Plus_2b489658.pdf']


['samples\\karta_info_skargi_wnioski_sn_233c5c16.pdf',
 'samples\\Mapa_Otwartych_Zasobow_Edukacyjnych_93594bec.pdf',
 'samples\\OC_os_fiz_przy_EDU_Plus_2b489658.pdf']

In [4]:
def extract_text_from_pdfs(pdf_paths):
    pdfs_texts = []
    for pdf_path in pdf_paths:
        full_pdf_text = ""

        print(f"Text extraction from: {os.path.basename(pdf_path)}\n")
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                temp_text = page.extract_text(layout=True)
                full_pdf_text += temp_text + "\n"
        pdfs_texts.append(" ".join(full_pdf_text.split()))
        print(f"Extracted text from {os.path.basename(pdf_path)}: {len(full_pdf_text)} characters\n")

    print("Check of first 300 characters of the first PDF text:\n")
    preview = pdfs_texts[0][:300] if pdfs_texts else ""
    print(preview)
    return pdfs_texts

In [ ]:
pdfs_texts = extract_text_from_pdfs(pdf_paths)

Text extraction from: karta_info_skargi_wnioski_sn_233c5c16.pdf

Extracted text from karta_info_skargi_wnioski_sn_233c5c16.pdf: 11780 characters

Text extraction from: Mapa_Otwartych_Zasobow_Edukacyjnych_93594bec.pdf

Extracted text from Mapa_Otwartych_Zasobow_Edukacyjnych_93594bec.pdf: 214466 characters

Text extraction from: OC_os_fiz_przy_EDU_Plus_2b489658.pdf

Extracted text from OC_os_fiz_przy_EDU_Plus_2b489658.pdf: 156246 characters

Check of first 300 characters of the first PDF text:

KARTA INFORMACYJNA Poniżej znajdą Państwo niezbędne informacje dotyczące przetwarzania danych osobowych w ramach rozpatrywania wniosków o udostępnienie informacji publicznej przez organy Sądu Najwyższego. Administratorem danych osobowych przetwarzanych przy rozpatrywaniu wniosków o dostęp Kto jest a


In [6]:
import re

def tokenize_text(pdfs_texts):
    token_pattern = re.compile(
    r"\d{2}-\d{3}|[\w]+|[^\w\s]",
    flags=re.UNICODE
    )
    pdfs_tokens = []
    for pdf_text in pdfs_texts:
        pdf_tokens = []

        for match in token_pattern.finditer(pdf_text):
            pdf_tokens.append({
                "token": match.group(),
                "start": match.start(),
                "end": match.end()
            })

        pdfs_tokens.append(pdf_tokens)

    return pdfs_tokens

In [7]:
pdfs_tokens = tokenize_text(pdfs_texts)
pdfs_tokens[0][420:450]  # Display the first 10 tokens of the first PDF

[{'token': 'że', 'start': 2803, 'end': 2805},
 {'token': 'SN', 'start': 2806, 'end': 2808},
 {'token': 'wykonał', 'start': 2809, 'end': 2816},
 {'token': 'dane', 'start': 2817, 'end': 2821},
 {'token': 'osobowe', 'start': 2822, 'end': 2829},
 {'token': '?', 'start': 2829, 'end': 2830},
 {'token': 'ciążące', 'start': 2831, 'end': 2838},
 {'token': 'na', 'start': 2839, 'end': 2841},
 {'token': 'nim', 'start': 2842, 'end': 2845},
 {'token': 'obowiązki', 'start': 2846, 'end': 2855},
 {'token': '.', 'start': 2855, 'end': 2856},
 {'token': 'Jako', 'start': 2857, 'end': 2861},
 {'token': 'administrator', 'start': 2862, 'end': 2875},
 {'token': 'danych', 'start': 2876, 'end': 2882},
 {'token': ',', 'start': 2882, 'end': 2883},
 {'token': 'SN', 'start': 2884, 'end': 2886},
 {'token': 'zapewnia', 'start': 2887, 'end': 2895},
 {'token': 'Państwu', 'start': 2896, 'end': 2903},
 {'token': 'możliwość', 'start': 2904, 'end': 2913},
 {'token': 'potwierdzenia', 'start': 2914, 'end': 2927},
 {'token': '

In [8]:
#W tej celli musimy określić chunk size, overlap i powinnismy dostać dict z 3 rzeczami 1. text, 2. start wdg znaków, 3. end wdg znaków. Jedendict jeden chunk, dicty w listcie [{text:STRING , start:INT , end: INT}]
def create_chunks(
    text: str,
    tokens: list[dict],
    chunk_size: int = 180,
    overlap: int = 30
) -> list[dict]:
    
    if not tokens:
        return []

    step = chunk_size - overlap
    chunks = []

    for token_start_index in range(0, len(tokens), step):
        token_end_index = min(
            token_start_index + chunk_size,
            len(tokens)
        )

        chunk_tokens = tokens[token_start_index:token_end_index]

        if not chunk_tokens:
            break

        chunk_start = chunk_tokens[0]["start"]
        chunk_end = chunk_tokens[-1]["end"]

        chunks.append({
            "text": text[chunk_start:chunk_end],
            "start": chunk_start,
            "end": chunk_end
        })

        if token_end_index >= len(tokens):
            break

    return chunks

In [9]:
#Dla powyzszej funkcji zrobic tak zeby argumnet wejsciowy to była cała tabela wszystkich pdfów, nie jeden pojedynczy. 
# Zgodność z poprzednimi funkcjami, które zwracają listę tekstów i listę tokenów dla wszystkich PDF-ów. 
# Wtedy funkcja create_chunks powinna iterować przez wszystkie teksty i tokeny, tworząc chunk dla każdego PDF-a i zwracając listę list chunków.
chunks = create_chunks(
    text=pdfs_texts[0],
    tokens=pdfs_tokens[0]
)

print(chunks[2])

{'text': 'w jakiś informacji profili preferencji osób, których dane dotyczą. jeszcze sposób? Dane gromadzone w ramach rozpatrywania wniosków o udostępnienie informacji publicznej Komu przekazywane są przez organy Sądu Najwyższego mogą być przekazywane wyłącznie uprawnionym organom, moje dane osobowe? w tym sądom administracyjnym i organom ścigania. Czy moje dane są Dane osobowe gromadzone w ramach rozpatrywania wniosków o udostępnienie informacji przekazywane poza Unię publicznej przez organy Sądu Najwyższego nie będą przekazywane poza teren Unii Europejską? Europejskiej. Dane osobowe gromadzone w ramach rozpatrywania wniosków o udostępnienie informacji Przez jaki czas publicznej przez organy Sądu Najwyższego przechowywane są bezterminowo. Jest to bowiem przetwarzane są moje niezbędne z punktu widzenia możliwości wykazania przed właściwymi organami, że SN wykonał dane osobowe? ciążące na nim obowiązki. Jako administrator danych, SN zapewnia Państwu możliwość potwierdzenia, czy w Sądzie

In [10]:
'''def create_chunks_with_metadata(
    text: str,
    tokens: list[dict],
    chunk_size: int = 180,
    overlap: int = 30
) -> tuple[list[dict], list[str]]:
    """
    Dzieli tokeny na chunki z overlapem.

    Zwraca:
    1. chunks_metadata - chunki wraz z metadanymi i tokenami,
    2. chunks_text - same teksty chunków.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size musi być większy od 0.")

    if overlap < 0:
        raise ValueError("overlap nie może być ujemny.")

    if overlap >= chunk_size:
        raise ValueError("overlap musi być mniejszy niż chunk_size.")

    step = chunk_size - overlap

    chunks_metadata = []
    chunks_text = []

    for chunk_id, token_start_index in enumerate(
        range(0, len(tokens), step)
    ):
        token_end_index = min(
            token_start_index + chunk_size,
            len(tokens)
        )

        chunk_tokens = tokens[token_start_index:token_end_index]

        if not chunk_tokens:
            break

        text_start = chunk_tokens[0]["start"]
        text_end = chunk_tokens[-1]["end"]

        # Wycięcie tekstu bezpośrednio z oryginalnego dokumentu
        chunk_text = text[text_start:text_end]

        chunk_metadata = {
            "chunk_id": chunk_id,

            # Zakres tokenów w całej liście tokenów
            "token_start_index": token_start_index,
            "token_end_index": token_end_index,

            # Zakres znaków w oryginalnym tekście
            "text_start": text_start,
            "text_end": text_end,

            # Tokeny należące do chunka
            "tokens": chunk_tokens,

            # Gotowy tekst do przekazania dalej
            "text": chunk_text
        }

        chunks_metadata.append(chunk_metadata)
        chunks_text.append(chunk_text)

        if token_end_index >= len(tokens):
            break

    return chunks_metadata, chunks_text'''

'def create_chunks_with_metadata(\n    text: str,\n    tokens: list[dict],\n    chunk_size: int = 180,\n    overlap: int = 30\n) -> tuple[list[dict], list[str]]:\n    """\n    Dzieli tokeny na chunki z overlapem.\n\n    Zwraca:\n    1. chunks_metadata - chunki wraz z metadanymi i tokenami,\n    2. chunks_text - same teksty chunków.\n    """\n\n    if chunk_size <= 0:\n        raise ValueError("chunk_size musi być większy od 0.")\n\n    if overlap < 0:\n        raise ValueError("overlap nie może być ujemny.")\n\n    if overlap >= chunk_size:\n        raise ValueError("overlap musi być mniejszy niż chunk_size.")\n\n    step = chunk_size - overlap\n\n    chunks_metadata = []\n    chunks_text = []\n\n    for chunk_id, token_start_index in enumerate(\n        range(0, len(tokens), step)\n    ):\n        token_end_index = min(\n            token_start_index + chunk_size,\n            len(tokens)\n        )\n\n        chunk_tokens = tokens[token_start_index:token_end_index]\n\n        if not 

In [11]:
'''chunks_metadata, chunks_text = create_chunks_with_metadata(
    text=pdfs_texts[0],
    tokens=pdfs_tokens[0],
    chunk_size=180,
    overlap=30
)
chunks_metadata[2], chunks_text[2]'''

'chunks_metadata, chunks_text = create_chunks_with_metadata(\n    text=pdfs_texts[0],\n    tokens=pdfs_tokens[0],\n    chunk_size=180,\n    overlap=30\n)\nchunks_metadata[2], chunks_text[2]'

In [18]:
#To co wyżej - klasyfikacja tokenów w chunku, z tym że obliczę od razu start i end w całym tekście, a nie w chunku. + walidacja na podstawie listy tokenów w całowym tekście czy lokalizacja sie zgadza
# Input: chunk(text, start, end)
#To samo co z funkccja dla chunków, potrzeba wrzucić wszsytkie pdfy i wtedy
from transformers import pipeline

def classify_tokens_in_chunk(chunk: dict) -> list[dict]:
   
   
    pipe = pipeline("token-classification", model="lexedit/herbert-polish-legal-ner", aggregation_strategy="simple")
    results = pipe(chunk['text'])

    # Przekształcenie wyników, aby uwzględnić start i end w całym tekście
    for result in results:
        result['start'] += chunk['start']
        result['end'] += chunk['start']

    # Przerobić walidacje: Tokeny są połączone (np. 'Sądu' 'Najwyższego' -> 'Sądu Najwyższego'), więc trzeba w walidacji rozdzielić tokeny z wyników klasyfikacji na pojedyncze tokeny i sprawdzić, czy każdy z nich pasuje do tokenów w całym tekście.
    for result in results:
        token_text = result['word'].split(' ', 1)[0]
        token_start = result['start']
        #token_end = result['end']

        # Znalezienie tokenu w pdfs_tokens, który odpowiada wynikowi klasyfikacji
        matching_tokens = [
            token for token in pdfs_tokens[0]  # Zakładamy, że analizujemy pierwszy PDF
            if token['start'] == token_start #and token['end'] == token_end
        ]

        if not matching_tokens:
            print(f"Token '{token_text}' na pozycji '{token_start}' z wyników klasyfikacji nie pasuje do żadnego tokenu w całym tekście.")
    return results



In [19]:
res = classify_tokens_in_chunk(chunks[0])
res

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity_group': 'ORG',
  'score': np.float32(0.5063702),
  'word': 'Sądu',
  'start': 187,
  'end': 191},
 {'entity_group': 'ORG',
  'score': np.float32(0.6692166),
  'word': 'Sądu',
  'start': 377,
  'end': 381},
 {'entity_group': 'ORG',
  'score': np.float32(0.97810936),
  'word': 'Sąd Najwyższy Rzeczypospolitej Polskiej',
  'start': 430,
  'end': 469},
 {'entity_group': 'LOC_PUB',
  'score': np.float32(0.95824784),
  'word': 'Warszawie',
  'start': 483,
  'end': 492},
 {'entity_group': 'LOC',
  'score': np.float32(0.9258698),
  'word': 'Placu Krasińskich 2 / 4 / 6',
  'start': 507,
  'end': 530},
 {'entity_group': 'ORG',
  'score': np.float32(0.54268837),
  'word': 'Sądu',
  'start': 816,
  'end': 820},
 {'entity_group': 'EMAIL',
  'score': np.float32(0.99791104),
  'word': 'iod @ sn . pl',
  'start': 953,
  'end': 962},
 {'entity_group': 'ORG',
  'score': np.float32(0.5658915),
  'word': 'Sądu',
  'start': 1109,
  'end': 1113},
 {'entity_group': 'ORG',
  'score': np.float32(0.531

In [14]:
from transformers import AutoModelForTokenClassification

MODEL_NAME = "lexedit/herbert-polish-legal-ner"

model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

print(model.config.id2label)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{0: 'B-DATE', 1: 'B-DIAGNOSIS', 2: 'B-EMAIL', 3: 'B-HEALTH_FACILITY', 4: 'B-IBAN', 5: 'B-ID', 6: 'B-LOC', 7: 'B-LOC_PUB', 8: 'B-MEDICAL_ID', 9: 'B-MONEY', 10: 'B-ORG', 11: 'B-PER', 12: 'B-PHONE', 13: 'B-WATERMARK', 14: 'I-DATE', 15: 'I-DIAGNOSIS', 16: 'I-EMAIL', 17: 'I-HEALTH_FACILITY', 18: 'I-IBAN', 19: 'I-ID', 20: 'I-LOC', 21: 'I-LOC_PUB', 22: 'I-MEDICAL_ID', 23: 'I-MONEY', 24: 'I-ORG', 25: 'I-PER', 26: 'I-PHONE', 27: 'I-WATERMARK', 28: 'O'}


In [ ]:
#CONSTANTS
import json

with open("constants/PERSON_TITLES.json", encoding="utf-8") as f:
    PERSON_TITLES = json.load(f)["PERSON_TITLES"]
with open("constants/FORMYPRAWNEORG.json", encoding="utf-8") as f:
    FORMYPRAWNEORG = json.load(f)["FORMYPRAWNEORG"]
with open("constants/CITIES.json", encoding="utf-8") as f:
    CITIES = json.load(f)["CITIES"]


In [23]:
#def add_person_titles_to_bio(pdfs_tokens, ):
#    normalized_titles = {title.lower().rstrip(".") for title in PERSON_TITLES}
#
#    for pdf_tokens in pdfs_tokens:
#        for i in range(1, len(pdf_tokens)):
#            current_token = pdf_tokens[i]
#            previous_token = pdf_tokens[i - 1]
#
#            if current_token.get("label") != "B-PERSON":
#                continue
#
#            previous_text = previous_token["token"].lower().rstrip(".")
#
#            if previous_text in normalized_titles:
#                previous_token["label"] = "B-PERSON"
#                current_token["label"] = "I-PERSON"
#
#    return pdfs_tokens

In [ ]:
check_pdfs_tokens = [
    [
        {"token": "Spotkanie", "start": 0, "end": 9, "label": "O"},
        {"token": "prowadziła", "start": 10, "end": 20, "label": "O"},
        {"token": "dr", "start": 21, "end": 23, "label": "O"},
        {"token": ".", "start": 23, "end": 24, "label": "O"},
        {"token": "Karolina", "start": 25, "end": 33, "label": "B-PERSON"},
        {"token": "Grodecka", "start": 34, "end": 42, "label": "I-PERSON"},
        {"token": ".", "start": 42, "end": 43, "label": "O"},
    ],
    [
        {"token": "Opinię", "start": 0, "end": 6, "label": "O"},
        {"token": "wydał", "start": 7, "end": 12, "label": "O"},
        {"token": "prof", "start": 13, "end": 17, "label": "O"},
        {"token": ".", "start": 17, "end": 18, "label": "O"},
        {"token": "dr", "start": 19, "end": 21, "label": "O"},
        {"token": ".", "start": 21, "end": 22, "label": "O"},
        {"token": "hab", "start": 23, "end": 26, "label": "O"},
        {"token": ".", "start": 26, "end": 27, "label": "O"},
        {"token": "Jan", "start": 28, "end": 31, "label": "B-PERSON"},
        {"token": "Kowalski", "start": 32, "end": 40, "label": "I-PERSON"},
    ],
    [
        {"token": "Szanowny", "start": 0, "end": 8, "label": "O"},
        {"token": "doktorze", "start": 9, "end": 17, "label": "O"},
        {"token": "habilitowany", "start": 18, "end": 31, "label": "O"},
        {"token": "Piotr", "start": 32, "end": 37, "label": "B-PERSON"},
        {"token": "Nowak", "start": 38, "end": 43, "label": "I-PERSON"},
    ],
    [
        {"token": "To", "start": 0, "end": 2, "label": "O"},
        {"token": "nie", "start": 3, "end": 6, "label": "O"},
        {"token": "jest", "start": 7, "end": 11, "label": "O"},
        {"token": "tytuł", "start": 12, "end": 17, "label": "O"},
        {"token": "Anna", "start": 18, "end": 22, "label": "B-PERSON"},
        {"token": "Wiśniewska", "start": 23, "end": 33, "label": "I-PERSON"},
    ],
]

In [25]:
check_pdfs_tokens

[[{'token': 'Spotkanie', 'start': 0, 'end': 9, 'label': 'O'},
  {'token': 'prowadziła', 'start': 10, 'end': 20, 'label': 'O'},
  {'token': 'dr', 'start': 21, 'end': 23, 'label': 'O'},
  {'token': '.', 'start': 23, 'end': 24, 'label': 'O'},
  {'token': 'Karolina', 'start': 25, 'end': 33, 'label': 'B-PERSON'},
  {'token': 'Grodecka', 'start': 34, 'end': 42, 'label': 'I-PERSON'},
  {'token': '.', 'start': 42, 'end': 43, 'label': 'O'}],
 [{'token': 'Opinię', 'start': 0, 'end': 6, 'label': 'O'},
  {'token': 'wydał', 'start': 7, 'end': 12, 'label': 'O'},
  {'token': 'prof', 'start': 13, 'end': 17, 'label': 'O'},
  {'token': '.', 'start': 17, 'end': 18, 'label': 'O'},
  {'token': 'dr', 'start': 19, 'end': 21, 'label': 'O'},
  {'token': '.', 'start': 21, 'end': 22, 'label': 'O'},
  {'token': 'hab', 'start': 23, 'end': 26, 'label': 'O'},
  {'token': '.', 'start': 26, 'end': 27, 'label': 'O'},
  {'token': 'Jan', 'start': 28, 'end': 31, 'label': 'B-PERSON'},
  {'token': 'Kowalski', 'start': 32, '

In [26]:
def add_person_titles_to_bio(check_pdfs_tokens, person_titles):
    titles = {title.lower().rstrip(".") for title in person_titles}

    for pdf_tokens in check_pdfs_tokens:
        for i, token in enumerate(pdf_tokens):
            if token.get("label") != "B-PERSON":
                continue

            title_indexes = []
            title_found = False
            non_title_before_name = 0
            j = i - 1

            while j >= 0:
                text = pdf_tokens[j]["token"]
                normalized = text.lower().rstrip(".")

                if text == ".":
                    title_indexes.append(j)
                    j -= 1
                    continue

                if normalized in titles:
                    title_indexes.append(j)
                    title_found = True
                    j -= 1
                    continue

                if title_found:
                    break

                non_title_before_name += 1

                if non_title_before_name == 2:
                    break

                title_indexes.append(j)
                j -= 1

            if not title_found:
                continue

            title_indexes.reverse()

            pdf_tokens[title_indexes[0]]["label"] = "B-PERSON"

            for title_index in title_indexes[1:]:
                pdf_tokens[title_index]["label"] = "I-PERSON"

            pdf_tokens[i]["label"] = "I-PERSON"

    return check_pdfs_tokens

In [28]:
check_pdfs_tokens2 = add_person_titles_to_bio(check_pdfs_tokens, PERSON_TITLES)

check_pdfs_tokens2

[[{'token': 'Spotkanie', 'start': 0, 'end': 9, 'label': 'O'},
  {'token': 'prowadziła', 'start': 10, 'end': 20, 'label': 'O'},
  {'token': 'dr', 'start': 21, 'end': 23, 'label': 'B-PERSON'},
  {'token': '.', 'start': 23, 'end': 24, 'label': 'I-PERSON'},
  {'token': 'Karolina', 'start': 25, 'end': 33, 'label': 'I-PERSON'},
  {'token': 'Grodecka', 'start': 34, 'end': 42, 'label': 'I-PERSON'},
  {'token': '.', 'start': 42, 'end': 43, 'label': 'O'}],
 [{'token': 'Opinię', 'start': 0, 'end': 6, 'label': 'O'},
  {'token': 'wydał', 'start': 7, 'end': 12, 'label': 'O'},
  {'token': 'prof', 'start': 13, 'end': 17, 'label': 'B-PERSON'},
  {'token': '.', 'start': 17, 'end': 18, 'label': 'I-PERSON'},
  {'token': 'dr', 'start': 19, 'end': 21, 'label': 'I-PERSON'},
  {'token': '.', 'start': 21, 'end': 22, 'label': 'I-PERSON'},
  {'token': 'hab', 'start': 23, 'end': 26, 'label': 'I-PERSON'},
  {'token': '.', 'start': 26, 'end': 27, 'label': 'I-PERSON'},
  {'token': 'Jan', 'start': 28, 'end': 31, 'lab

In [29]:
check_org_pdfs_tokens = [
    [
        {"token": "Umowę", "start": 0, "end": 5, "label": "O"},
        {"token": "podpisała", "start": 6, "end": 15, "label": "O"},
        {"token": "ABC", "start": 16, "end": 19, "label": "B-ORGANIZATION"},
        {"token": "sp", "start": 20, "end": 22, "label": "O"},
        {"token": ".", "start": 22, "end": 23, "label": "O"},
        {"token": "z", "start": 24, "end": 25, "label": "O"},
        {"token": "o", "start": 26, "end": 27, "label": "O"},
        {"token": ".", "start": 27, "end": 28, "label": "O"},
        {"token": "o", "start": 28, "end": 29, "label": "O"},
        {"token": ".", "start": 29, "end": 30, "label": "O"},
    ],
    [
        {"token": "Akcje", "start": 0, "end": 5, "label": "O"},
        {"token": "wyemitowała", "start": 6, "end": 17, "label": "O"},
        {"token": "spółka", "start": 18, "end": 24, "label": "O"},
        {"token": "akcyjna", "start": 25, "end": 32, "label": "O"},
        {"token": "Polenergia", "start": 33, "end": 43, "label": "B-ORGANIZATION"},
    ],
    [
        {"token": "Raport", "start": 0, "end": 6, "label": "O"},
        {"token": "opublikowała", "start": 7, "end": 19, "label": "O"},
        {"token": "Fundacja", "start": 20, "end": 28, "label": "O"},
        {"token": "Dajemy", "start": 29, "end": 35, "label": "B-ORGANIZATION"},
        {"token": "Dzieciom", "start": 36, "end": 44, "label": "I-ORGANIZATION"},
        {"token": "Siłę", "start": 45, "end": 49, "label": "I-ORGANIZATION"},
    ],
    [
        {"token": "Partnerem", "start": 0, "end": 9, "label": "O"},
        {"token": "jest", "start": 10, "end": 14, "label": "O"},
        {"token": "Microsoft", "start": 15, "end": 24, "label": "B-ORGANIZATION"},
        {"token": "Corporation", "start": 25, "end": 36, "label": "I-ORGANIZATION"},
        {"token": ".", "start": 36, "end": 37, "label": "O"},
    ],
]

In [35]:
import copy

def is_legal_form_token(token, legal_forms):
    return token.lower().rstrip(".") in legal_forms


def is_legal_form_dot(tokens, index, legal_forms):
    if tokens[index]["token"] != ".":
        return False

    prev_is_form = (
        index - 1 >= 0
        and is_legal_form_token(tokens[index - 1]["token"], legal_forms)
    )

    next_is_form = (
        index + 1 < len(tokens)
        and is_legal_form_token(tokens[index + 1]["token"], legal_forms)
    )

    return prev_is_form or next_is_form


def add_legal_forms_to_organization_bio(check_org_pdfs_tokens, FORMYPRAWNEORG):
    result_tokens = copy.deepcopy(check_org_pdfs_tokens)

    legal_forms = {
        form.lower().rstrip(".")
        for form in FORMYPRAWNEORG
    }

    for pdf_tokens in result_tokens:
        i = 0

        while i < len(pdf_tokens):
            if pdf_tokens[i].get("label") != "B-ORGANIZATION":
                i += 1
                continue

            org_start = i
            org_end = i

            while (
                org_end + 1 < len(pdf_tokens)
                and pdf_tokens[org_end + 1].get("label") == "I-ORGANIZATION"
            ):
                org_end += 1

            form_indexes = []
            j = org_start - 1

            while j >= 0:
                text = pdf_tokens[j]["token"]

                if is_legal_form_token(text, legal_forms):
                    form_indexes.append(j)
                    j -= 1
                    continue

                if is_legal_form_dot(pdf_tokens, j, legal_forms):
                    form_indexes.append(j)
                    j -= 1
                    continue

                break

            if form_indexes:
                form_indexes.reverse()

                pdf_tokens[form_indexes[0]]["label"] = "B-ORGANIZATION"

                for form_index in form_indexes[1:]:
                    pdf_tokens[form_index]["label"] = "I-ORGANIZATION"

                pdf_tokens[org_start]["label"] = "I-ORGANIZATION"
                org_start = form_indexes[0]

            j = org_end + 1

            while j < len(pdf_tokens):
                text = pdf_tokens[j]["token"]

                if is_legal_form_token(text, legal_forms):
                    pdf_tokens[j]["label"] = "I-ORGANIZATION"
                    org_end = j
                    j += 1
                    continue

                if is_legal_form_dot(pdf_tokens, j, legal_forms):
                    pdf_tokens[j]["label"] = "I-ORGANIZATION"
                    org_end = j
                    j += 1
                    continue

                break

            i = org_end + 1

    return result_tokens

In [36]:
checked_org_tokens = add_legal_forms_to_organization_bio(
    check_org_pdfs_tokens,
    FORMYPRAWNEORG
)
checked_org_tokens

[[{'token': 'Umowę', 'start': 0, 'end': 5, 'label': 'O'},
  {'token': 'podpisała', 'start': 6, 'end': 15, 'label': 'O'},
  {'token': 'ABC', 'start': 16, 'end': 19, 'label': 'B-ORGANIZATION'},
  {'token': 'sp', 'start': 20, 'end': 22, 'label': 'I-ORGANIZATION'},
  {'token': '.', 'start': 22, 'end': 23, 'label': 'I-ORGANIZATION'},
  {'token': 'z', 'start': 24, 'end': 25, 'label': 'I-ORGANIZATION'},
  {'token': 'o', 'start': 26, 'end': 27, 'label': 'I-ORGANIZATION'},
  {'token': '.', 'start': 27, 'end': 28, 'label': 'I-ORGANIZATION'},
  {'token': 'o', 'start': 28, 'end': 29, 'label': 'I-ORGANIZATION'},
  {'token': '.', 'start': 29, 'end': 30, 'label': 'I-ORGANIZATION'}],
 [{'token': 'Akcje', 'start': 0, 'end': 5, 'label': 'O'},
  {'token': 'wyemitowała', 'start': 6, 'end': 17, 'label': 'O'},
  {'token': 'spółka', 'start': 18, 'end': 24, 'label': 'B-ORGANIZATION'},
  {'token': 'akcyjna', 'start': 25, 'end': 32, 'label': 'I-ORGANIZATION'},
  {'token': 'Polenergia', 'start': 33, 'end': 43, '

In [ ]:
check_city_pdfs_tokens = [
    [
        {"token": "Spotkanie", "start": 0, "end": 9, "label": "O"},
        {"token": "odbyło", "start": 10, "end": 16, "label": "O"},
        {"token": "się", "start": 17, "end": 20, "label": "O"},
        {"token": "w", "start": 21, "end": 22, "label": "O"},
        {"token": "Poznaniu", "start": 23, "end": 31, "label": "O"},
        {"token": ".", "start": 31, "end": 32, "label": "O"},
    ],
    [
        {"token": "Sąd", "start": 0, "end": 3, "label": "B-CITY"},
        {"token": "Najwyższy", "start": 4, "end": 13, "label": "I-CITY"},
        {"token": "ma", "start": 14, "end": 16, "label": "O"},
        {"token": "siedzibę", "start": 17, "end": 25, "label": "O"},
        {"token": "w", "start": 26, "end": 27, "label": "O"},
        {"token": "Warszawie", "start": 28, "end": 37, "label": "B-CITY"},
    ],
    [
        {"token": "Delegatura", "start": 0, "end": 10, "label": "O"},
        {"token": "działa", "start": 11, "end": 17, "label": "O"},
        {"token": "w", "start": 18, "end": 19, "label": "O"},
        {"token": "Biała", "start": 20, "end": 25, "label": "B-CITY"},
        {"token": "Podlaska", "start": 26, "end": 34, "label": "I-CITY"},
    ],
]

In [53]:
import copy

def remove_invalid_city_tokens(pdfs_tokens, CITIES):
    result_tokens = copy.deepcopy(pdfs_tokens)

    cities = {city.lower() for city in CITIES}

    for pdf_tokens in result_tokens:
        i = 0

        while i < len(pdf_tokens):
            if pdf_tokens[i].get("label") != "B-CITY":
                i += 1
                continue

            start = i
            end = i

            while (
                end + 1 < len(pdf_tokens)
                and pdf_tokens[end + 1].get("label") == "I-CITY"
            ):
                end += 1

            city_name = " ".join(
                token["token"] for token in pdf_tokens[start:end + 1]
            ).lower()

            if city_name not in cities:
                for j in range(start, end + 1):
                    pdf_tokens[j]["label"] = "O"

            i = end + 1

    return result_tokens

def mark_city_tokens(pdfs_tokens, CITIES):
    cities = {city.lower() for city in CITIES}

    for pdf_tokens in pdfs_tokens:
        for token in pdf_tokens:
            if token["token"].lower() in cities:
                token["label"] = "B-CITY"

    return pdfs_tokens

In [55]:
checked_city_tokens = remove_invalid_city_tokens(
    check_city_pdfs_tokens,
    CITIES
)
checked_city_tokens = mark_city_tokens(checked_city_tokens, CITIES)
checked_city_tokens

[[{'token': 'Spotkanie', 'start': 0, 'end': 9, 'label': 'O'},
  {'token': 'odbyło', 'start': 10, 'end': 16, 'label': 'O'},
  {'token': 'się', 'start': 17, 'end': 20, 'label': 'O'},
  {'token': 'w', 'start': 21, 'end': 22, 'label': 'O'},
  {'token': 'Poznaniu', 'start': 23, 'end': 31, 'label': 'B-CITY'},
  {'token': '.', 'start': 31, 'end': 32, 'label': 'O'}],
 [{'token': 'Sąd', 'start': 0, 'end': 3, 'label': 'O'},
  {'token': 'Najwyższy', 'start': 4, 'end': 13, 'label': 'O'},
  {'token': 'ma', 'start': 14, 'end': 16, 'label': 'O'},
  {'token': 'siedzibę', 'start': 17, 'end': 25, 'label': 'O'},
  {'token': 'w', 'start': 26, 'end': 27, 'label': 'O'},
  {'token': 'Warszawie', 'start': 28, 'end': 37, 'label': 'B-CITY'}],
 [{'token': 'Delegatura', 'start': 0, 'end': 10, 'label': 'O'},
  {'token': 'działa', 'start': 11, 'end': 17, 'label': 'O'},
  {'token': 'w', 'start': 18, 'end': 19, 'label': 'O'},
  {'token': 'Biała', 'start': 20, 'end': 25, 'label': 'O'},
  {'token': 'Podlaska', 'start':